# NC-Musical: AI Music Transcription (Batch Colab GPU Pipeline)
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine directly inside Google Colab.

**Hardware Acceleration:** NVIDIA GPUs (T4, L4, A100)

**Key Features:**
- **Batch Transcription:** Process single or multiple YouTube URLs / uploaded files sequentially.
- **Original Song Naming:** Automatically names MIDI and chord files after the original YouTube song title or uploaded filename.
- **Dynamic Prompts:** No hardcoded links — easily paste links or be prompted on run.
- **Future-Proof yt-dlp:** Direct live master-branch builds for dependable extraction.
- **SoundFont Previews:** In-notebook audio playback via `MS Basic.sf3`.
- **Automated Chord Analysis:** Extracts and timestamps harmonic progressions using `music21`.
- **Automatic Downloads & ZIP:** Triggers immediate browser downloads and bundles batch results into `.zip`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [ ]:
# @title 1. (Optional) Connect Google Drive for Persistent Model Storage
# @markdown Connecting Google Drive saves downloaded AI models and SoundFonts permanently so you don't re-download them across Colab restarts.
# @markdown Set **use_google_drive** to `False` to skip the authorization prompt and use fast session temporary storage instead.
use_google_drive = True # @param {type:"boolean"}
import os

if use_google_drive:
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive is already connected and active!")
    else:
        try:
            from google.colab import drive
            print("Connecting to Google Drive...")
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"Notice: Google Drive mount skipped ({e}). Falling back to local session storage.")

if os.path.exists('/content/drive/MyDrive'):
    drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
    os.makedirs(drive_models_dir, exist_ok=True)
    hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")
    print(f"✓ Persistent Model Storage enabled: {drive_models_dir}")
else:
    local_cache_dir = "/content/cache/huggingface"
    os.makedirs(local_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = local_cache_dir
    print("ℹ Session Storage active: Models will be cached in /content/cache/ for this session.")


In [ ]:
# @title 2. Check Hardware (GPU / TPU v5e-1), Tokens & Setup Repository
import os
import sys
import subprocess
import getpass
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Checking hardware accelerator environment...")
has_gpu = torch.cuda.is_available()
has_tpu = False

if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ NVIDIA GPU Detected: {gpu_name}")
    !nvidia-smi
else:
    # Check for Google Colab TPU v5e / TPU v2/v3 environment
    if 'COLAB_TPU_ADDR' in os.environ or os.environ.get('PJRT_DEVICE') == 'TPU' or os.path.exists('/dev/accel0'):
        has_tpu = True
        os.environ['PJRT_DEVICE'] = 'TPU'
        print('✓ Google Cloud TPU (v5e-1 / PJRT) Detected!')
    else:
        try:
            import torch_xla.core.xla_model as xm
            has_tpu = True
            print(f'✓ Google Cloud TPU Detected: {xm.xla_device()}')
        except Exception:
            print('ℹ Standard CPU environment detected (No GPU/TPU found).')

# Configure HuggingFace Token (HF_TOKEN)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    print("\nNotice: MuScriptor models on HuggingFace require a HuggingFace User Token (HF_TOKEN).")
    print("If saved in Colab Secrets as 'HF_TOKEN', it is loaded automatically. Otherwise, enter below:")
    try:
        hf_token = getpass.getpass("HuggingFace Token (HF_TOKEN): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace Access Token configured successfully!")

print("\nInstalling system dependencies (FFmpeg and FluidSynth)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth

print("\nCloning / Updating NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
        
        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")
        
        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    !git pull || true
    print("\nInstalling Python dependencies (Future-proof live yt-dlp master branch)...")
    !pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" music21 soundfile
    if has_tpu:
        try:
            import torch_xla
        except ImportError:
            print("Installing torch_xla TPU support...")
            !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html
    !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


In [ ]:
# @title 3. Setup MS Basic.sf3 SoundFont
import os
import shutil
import urllib.request

local_sf3_path = "/content/NC-Musical/MS Basic.sf3"
drive_sf3_path = "/content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3"
has_drive = os.path.exists("/content/drive/MyDrive")

target_sf3 = drive_sf3_path if has_drive else local_sf3_path

if not os.path.exists(target_sf3):
    print("Downloading MS Basic.sf3 SoundFont (~50MB)...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            os.makedirs(os.path.dirname(target_sf3), exist_ok=True)
            urllib.request.urlretrieve(url, target_sf3)
            print(f"✓ MS Basic.sf3 downloaded successfully to {target_sf3}!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print(f"✓ MS Basic.sf3 SoundFont found at: {target_sf3}")

# Sync to local workspace path if Drive was used
if has_drive and os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    shutil.copy2(drive_sf3_path, local_sf3_path)


In [ ]:
# @title 4. AI Music Transcription (Batch & Single Processing)
# @markdown ### 📥 Audio Input & Model Configuration
input_source = "YouTube URL(s)" # @param ["YouTube URL(s)", "Upload Audio/Video File(s)"]
youtube_urls = "" # @param {type:"string"}
model_size = "large" # @param ["small", "medium", "large"]

# @markdown ### 🎛️ Target Instrument Focus (Preset Mode)
# @markdown Choose which instrument(s) to isolate and transcribe from the mix:
instrument_focus = "All Instruments (Full Mix)" # @param ["All Instruments (Full Mix)", "Piano Only (Acoustic & Electric Keys)", "Guitars (Acoustic & Electric)", "Rhythm Section (Bass & Drums)", "Vocals & Lead Melody", "Strings & Orchestral", "Custom Selection (Configure Below)"]

# @markdown ### ⚙️ Advanced Custom Instrument Filters (Used if 'Custom Selection' chosen above)
focus_piano = True # @param {type:"boolean"}
focus_acoustic_guitar = False # @param {type:"boolean"}
focus_electric_guitar = False # @param {type:"boolean"}
focus_bass = False # @param {type:"boolean"}
focus_drums = False # @param {type:"boolean"}
focus_vocals = False # @param {type:"boolean"}
focus_strings = False # @param {type:"boolean"}
focus_brass = False # @param {type:"boolean"}
focus_woodwinds = False # @param {type:"boolean"}
focus_synths = False # @param {type:"boolean"}
custom_instruments_text = "" # @param {type:"string"}

# @markdown ### 📤 Export Options
export_chord_names = True # @param {type:"boolean"}
download_zip_for_batch = True # @param {type:"boolean"}

import os
import sys
import re
import time
import gc
import zipfile
import shutil
import torch
from pathlib import Path
from IPython.display import Audio, display
import google.colab.files

# Set PyTorch memory allocator to avoid fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Always fetch latest master build of yt-dlp for future-proof YouTube extraction
!pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" music21
import yt_dlp

def sanitize_filename(name: str) -> str:
    """Clean filename of illegal filesystem characters."""
    clean = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    clean = re.sub(r'\s+', " ", clean)
    return clean[:120] if clean else "transcription_result"

# ── 1. Resolve Target Instruments ─────────────────────────────────────────────
INST_ALIASES = {
    "piano": "acoustic_piano", "guitar": "acoustic_guitar",
    "electric_guitar": "clean_electric_guitar", "bass": "electric_bass",
    "drums": "drums", "drum": "drums", "percussion": "drums",
    "strings": "string_ensemble", "brass": "brass_section",
    "sax": "soprano_and_alto_sax", "saxophone": "soprano_and_alto_sax",
    "flute": "flutes", "synth": "synth_lead", "pad": "synth_pad",
    "vocal": "voice", "vocals": "voice",
}

inst_list = None
if instrument_focus == "All Instruments (Full Mix)":
    inst_list = None
elif instrument_focus == "Piano Only (Acoustic & Electric Keys)":
    inst_list = ["acoustic_piano", "electric_piano"]
elif instrument_focus == "Guitars (Acoustic & Electric)":
    inst_list = ["acoustic_guitar", "clean_electric_guitar", "distorted_electric_guitar"]
elif instrument_focus == "Rhythm Section (Bass & Drums)":
    inst_list = ["electric_bass", "acoustic_bass", "drums"]
elif instrument_focus == "Vocals & Lead Melody":
    inst_list = ["voice", "synth_lead"]
elif instrument_focus == "Strings & Orchestral":
    inst_list = ["string_ensemble", "brass_section", "flutes"]
elif instrument_focus == "Custom Selection (Configure Below)":
    selected = []
    if focus_piano:
        selected.extend(["acoustic_piano", "electric_piano"])
    if focus_acoustic_guitar:
        selected.append("acoustic_guitar")
    if focus_electric_guitar:
        selected.extend(["clean_electric_guitar", "distorted_electric_guitar"])
    if focus_bass:
        selected.extend(["electric_bass", "acoustic_bass"])
    if focus_drums:
        selected.append("drums")
    if focus_vocals:
        selected.append("voice")
    if focus_strings:
        selected.append("string_ensemble")
    if focus_brass:
        selected.append("brass_section")
    if focus_woodwinds:
        selected.append("flutes")
    if focus_synths:
        selected.extend(["synth_lead", "synth_pad"])
    
    if custom_instruments_text.strip():
        for raw_name in custom_instruments_text.split(","):
            k = raw_name.strip().lower().replace(" ", "_")
            if k:
                selected.append(INST_ALIASES.get(k, k))
    
    inst_list = list(dict.fromkeys(selected)) if selected else None

if inst_list:
    print(f"🎯 Target Instrument Focus: {inst_list}")
else:
    print("🎯 Target Instrument Focus: ALL Instruments (Full Multitrack Transcription)")

# ── 2. Gather Items into Processing Queue ─────────────────────────────────────
queue = [] # list of tuples: (item_type, source_path_or_url, song_title)

if input_source == "YouTube URL(s)":
    raw_input = youtube_urls.strip()
    if not raw_input:
        print("━" * 70)
        print("No YouTube URL provided in the form field.")
        raw_input = input("Please paste one or multiple YouTube URLs (separated by spaces or commas): ").strip()
        print("━" * 70 + "\n")
    
    urls = [u.strip() for u in re.split(r'[\n,\s]+', raw_input) if u.strip() and ("youtube.com" in u or "youtu.be" in u)]
    if not urls:
        print("❌ No valid YouTube URLs found. Please check your input and run again.")
    else:
        for url in urls:
            queue.append(("youtube", url, None))
else:
    print("Please click 'Choose Files' below to upload audio/video file(s) (MP3, WAV, MP4, M4A, FLAC):")
    uploaded = google.colab.files.upload()
    if uploaded:
        for fname in uploaded.keys():
            stem = Path(fname).stem
            queue.append(("file", fname, stem))
    else:
        print("❌ No files were uploaded.")

if not queue:
    print("Queue is empty. Nothing to transcribe.")
else:
    print(f"✓ Found {len(queue)} track(s) in queue.")

    # ── 3. Persistent Hardware & Model Management (OOM Prevention) ───────────
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
    
    global _CACHED_MODEL, _CACHED_MODEL_SIZE, _CACHED_DEVICE
    if '_CACHED_MODEL' not in globals():
        _CACHED_MODEL = None
        _CACHED_MODEL_SIZE = None
        _CACHED_DEVICE = None

    from muscriptor.transcription_model import TranscriptionModel
    from muscriptor.utils.audio import _read_wav_file
    from muscriptor.events import ProgressEvent
    from muscriptor.utils.auralization import synthesize

    # Check if model is already in memory
    if _CACHED_MODEL is not None and _CACHED_MODEL_SIZE == model_size and _CACHED_DEVICE == device:
        print(f"✓ Reusing loaded '{model_size}' model on {dev_name} (0s reload time, 0MB extra VRAM).")
        model = _CACHED_MODEL
    else:
        if _CACHED_MODEL is not None:
            print(f"Releasing previous '{_CACHED_MODEL_SIZE}' model from VRAM...")
            del _CACHED_MODEL
            _CACHED_MODEL = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        print(f"\nLoading MuScriptor AI Model ('{model_size}' on {dev_name})...")
        model = TranscriptionModel.load_model(weights_path=model_size, device=device)
        _CACHED_MODEL = model
        _CACHED_MODEL_SIZE = model_size
        _CACHED_DEVICE = device
        print("✓ Model loaded successfully!\n")

    # YouTube download helper
    def fetch_youtube_audio(url, temp_stem="/content/yt_temp"):
        cookie_paths = [
            "/content/cookies.txt",
            "/content/drive/MyDrive/cookies.txt",
            "/content/drive/MyDrive/NC-Musical-Models/cookies.txt",
        ]
        found_cookie = next((cp for cp in cookie_paths if os.path.exists(cp)), None)

        base_opts = {
            'format': 'bestaudio/best',
            'outtmpl': f'{temp_stem}.%(ext)s',
            'quiet': True,
            'no_warnings': True,
            'noplaylist': True,
        }

        strategies = [
            ("Standard yt-dlp", {}),
            ("Android + Web client", {'extractor_args': {'youtube': {'player_client': ['android', 'web']}}}),
            ("iOS + Mobile Web client", {'extractor_args': {'youtube': {'player_client': ['ios', 'mweb']}}}),
            ("TV + Embedded client", {'extractor_args': {'youtube': {'player_client': ['tv_embedded', 'web']}}}),
        ]
        if found_cookie:
            strategies.append(("Standard yt-dlp + Cookies", {'cookiefile': found_cookie}))
            strategies.append(("Android client + Cookies", {'cookiefile': found_cookie, 'extractor_args': {'youtube': {'player_client': ['android', 'web']}}}))

        last_err = None
        title = "Unknown Song"

        for name, extra_opts in strategies:
            opts = {**base_opts, **extra_opts}
            try:
                for f in Path("/content").glob("yt_temp.*"):
                    try: os.remove(f)
                    except: pass
                
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(url, download=True)
                    if info and 'title' in info:
                        title = info['title']

                dl_files = list(Path("/content").glob("yt_temp.*"))
                if dl_files:
                    return str(dl_files[0]), title, None
            except Exception as e:
                last_err = e

        return None, title, last_err

    # Progress bar helper
    def _fmt_time(s):
        return f"{int(s//60)}m {int(s%60):02d}s" if s >= 60 else f"{s:.0f}s"

    def _draw_progress(completed, total, elapsed, note_count):
        pct = int(completed / total * 100) if total > 0 else 0
        bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
        eta = (elapsed / max(completed, 1)) * (total - completed) if completed > 0 else 0
        print(f"\r  [{bar}] {pct:3d}%  |  Elapsed: {_fmt_time(elapsed)}  |  ETA: {_fmt_time(eta)}  |  Notes: {note_count}    ",
              end="", flush=True)

    # ── 4. Execute Batch Queue ────────────────────────────────────────────────
    batch_results = []
    output_dir = Path("/content/NC_Musical_Outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    for idx, (itype, isrc, ititle) in enumerate(queue, 1):
        print("═" * 70)
        print(f"🎵 [Track {idx}/{len(queue)}] Processing: {isrc if itype == 'youtube' else isrc}")
        print("═" * 70)

        wav_input = "/content/current_input.wav"
        if os.path.exists(wav_input):
            os.remove(wav_input)

        song_title = ititle
        if itype == "youtube":
            print(f"Downloading YouTube audio...")
            raw_audio, fetched_title, err = fetch_youtube_audio(isrc)
            if not raw_audio:
                print(f"❌ Failed to download audio for {isrc}: {err}")
                continue
            song_title = sanitize_filename(fetched_title)
            print(f"✓ Title: {song_title}")
            print("Converting to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(raw_audio), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])
            try: os.remove(raw_audio)
            except: pass
        else:
            song_title = sanitize_filename(ititle)
            print(f"✓ Title: {song_title}")
            print(f"Converting {isrc} to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(isrc), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])

        if not os.path.exists(wav_input):
            print(f"❌ Failed to prepare audio for {song_title}.")
            continue

        with open(wav_input, "rb") as f:
            wav, sr = _read_wav_file(f)

        print(f"Transcribing audio notes...")
        events = []
        start_t = time.time()
        devnull = open(os.devnull, 'w')
        real_stdout = sys.stdout
        try:
            with torch.inference_mode():
                for ev in model.transcribe((wav, sr), instruments=inst_list, batch_size=1):
                    sys.stdout = real_stdout
                    if isinstance(ev, ProgressEvent):
                        if ev.total > 0:
                            _draw_progress(ev.completed, ev.total, time.time() - start_t, len(events))
                    else:
                        events.append(ev)
                    sys.stdout = devnull
        finally:
            sys.stdout = real_stdout
            devnull.close()

        elapsed = time.time() - start_t
        print(f"\n\n✅ Complete in {_fmt_time(elapsed)} — {len(events)} note events extracted.")

        # Save MIDI with original song title
        midi_bytes = model.events_to_midi_bytes(iter(events))
        midi_path = output_dir / f"{song_title}.mid"
        with open(midi_path, "wb") as f:
            f.write(midi_bytes)
        print(f"💾 MIDI Saved: {midi_path.name}")

        # Chord Analysis with original song title
        chord_path = None
        if export_chord_names:
            chord_path = output_dir / f"{song_title}_chords.txt"
            try:
                from music21 import converter, chord as m21chord
                score = converter.parse(str(midi_path))
                chordified = score.chordify()
                lines = [f"CHORD ANALYSIS — {song_title}", "=" * 50]
                prev_label = None
                for c in chordified.flatten().getElementsByClass(m21chord.Chord):
                    offset_sec = float(c.offset)
                    ts = f"{int(offset_sec//60)}:{offset_sec%60:05.2f}"
                    try:
                        label = f"{c.root().name} {c.commonName}" if c.root() else " ".join(p.name for p in c.pitches)
                    except Exception:
                        label = " ".join(p.name for p in c.pitches)
                    if label != prev_label:
                        lines.append(f"[{ts}]  {label}")
                        prev_label = label
                with open(chord_path, "w", encoding="utf-8") as f:
                    f.write("\n".join(lines))
                print(f"🎼 Chords Saved: {chord_path.name}")
            except Exception as e:
                print(f"Chord analysis notice: {e}")

        # Audio preview
        preview_wav = "/content/preview.wav"
        sf3_path = "/content/NC-Musical/MS Basic.sf3"
        try:
            synthesize(str(midi_path), preview_wav, soundfont_path=sf3_path if os.path.exists(sf3_path) else None)
            print(f"🎧 Audio Preview ({song_title}):")
            display(Audio(preview_wav))
        except Exception as e:
            pass

        # Auto download individual files
        try:
            google.colab.files.download(str(midi_path))
            if chord_path and chord_path.exists():
                google.colab.files.download(str(chord_path))
        except Exception as e:
            print(f"Download notice: {e}")

        batch_results.append({
            "title": song_title,
            "midi": str(midi_path),
            "chords": str(chord_path) if chord_path and chord_path.exists() else None,
            "notes": len(events),
            "time": _fmt_time(elapsed)
        })

        # Free temporary audio memory and flush CUDA cache
        del wav, events
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── 5. Batch ZIP Package ──────────────────────────────────────────────────
    if len(batch_results) > 1 and download_zip_for_batch:
        zip_path = "/content/NC_Musical_Batch_Results.zip"
        print("\n" + "═" * 70)
        print("📦 Packaging all batch transcriptions into ZIP archive...")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for item in batch_results:
                if os.path.exists(item['midi']):
                    zipf.write(item['midi'], arcname=Path(item['midi']).name)
                if item['chords'] and os.path.exists(item['chords']):
                    zipf.write(item['chords'], arcname=Path(item['chords']).name)
        print(f"✓ ZIP archive created: {zip_path}")
        google.colab.files.download(zip_path)

    # ── 6. Summary Table ──────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("📊 BATCH TRANSCRIPTION SUMMARY")
    print("═" * 70)
    for r in batch_results:
        print(f" • {r['title']} | Notes: {r['notes']:,} | Time: {r['time']}")
    print(f"\nAll results saved to directory: {output_dir}")
